In [55]:
import numpy as np
import pandas as pd
import time
from scipy.sparse import csr_matrix
import seaborn as sns
import hypernetx as hnx
import matplotlib.pyplot as plt

pd.set_option("display.max_rows", 999)

In [56]:
from pathlib import Path

savedir = Path('./hypergraph/sc_GEM')
savedir.mkdir(parents=True, exist_ok=True)


In [57]:
df_expression = pd.read_csv('./data/sc_GEM/expression_data.csv')

In [58]:
df_expression.columns

Index(['Cell ID', 'ADAM33', 'AFP', 'AK123759', 'ALDH3A1 ', 'BMP7', 'CDH1',
       'CDH22', 'CDX2', 'CER1', 'CHL1', 'COL20A1', 'COL23A1', 'CYBRD1', 'DAZL',
       'DNMT3B', 'DNMT3L', 'DPPA3', 'EPHB3', 'FGF4', 'FOSL1', 'FOXD2', 'HAND1',
       'HLX-AS1', 'IGF2', 'JARID2', 'KCNQ2', 'LEFTY', 'LTBR', 'LUM',
       'LY86-AS1', 'MMP9', 'MYC', 'NANOG', 'NESTIN', 'NFATC1', 'NFIX',
       'NKX2-5', 'NXRA8', 'OCT4', 'OTX2', 'PIWIL1', 'PLEKHH3', 'PRDM14',
       'SALL4', 'SOX2', 'STON2P1', 'SULT1A1', 'TBX3', 'TFAP2A', 'TFCP2L1',
       'TGFBR2', 'TMEM173', 'TNNI3', 'UNC45A', 'WNT10B', 'ZFP42', 'ZIC3',
       'ZNF662', 'ZSCAN18'],
      dtype='object')

In [59]:
df_methy = pd.read_csv('./data/sc_GEM/methylation_data.csv')

In [60]:
df_methy.columns

Index(['Cell ID', 'ADAM33', 'AFP', 'AK123759', 'ALDH3A1 ', 'BMP7', 'CDH1',
       'CDH22', 'CDX2', 'CER1', 'CHL1', 'COL20A1', 'COL23A1', 'CYBRD1', 'DAZL',
       'DNMT3B', 'DNMT3L', 'DPPA3', 'EPHB3', 'FGF4', 'FOSL1', 'FOXD2', 'HAND1',
       'HLX-AS1', 'IGF2', 'JARID2', 'KCNQ2', 'LEFTY', 'LTBR', 'LUM',
       'LY86-AS1', 'MMP9', 'MYC', 'NANOG', 'NESTIN', 'NFATC1', 'NFIX',
       'NKX2-5', 'NXRA8', 'OCT4', 'OTX2', 'PIWIL1', 'PLEKHH3', 'PRDM14',
       'SALL4', 'SOX2', 'STON2P1', 'SULT1A1', 'TBX3', 'TFAP2A', 'TFCP2L1',
       'TGFBR2', 'TMEM173', 'TNNI3', 'UNC45A', 'WNT10B', 'ZFP42', 'ZIC3',
       'ZNF662', 'ZSCAN18'],
      dtype='object')

Create the union of features.

In [61]:
df_omics = df_expression.set_index('Cell ID').join(df_methy.set_index('Cell ID'), lsuffix='_expr', rsuffix='_methy')

Remove feature columns that are all NaNs.

In [62]:
df_omics_clean = df_omics.dropna(axis=1, how='all')

In [63]:
df_omics_clean.count()

AFP_expr          224
BMP7_expr         224
CDH1_expr         224
CDX2_expr         224
CER1_expr         224
DAZL_expr         224
DNMT3B_expr       224
DNMT3L_expr       224
FGF4_expr         224
FOSL1_expr        224
HAND1_expr        224
IGF2_expr         224
JARID2_expr       224
KCNQ2_expr        224
LEFTY_expr        224
LUM_expr          224
MMP9_expr         224
MYC_expr          224
NANOG_expr        224
NESTIN_expr       224
NKX2-5_expr       224
OCT4_expr         224
OTX2_expr         224
PRDM14_expr       224
SALL4_expr        224
SOX2_expr         224
TBX3_expr         224
TFAP2A_expr       224
TFCP2L1_expr      224
TGFBR2_expr       224
WNT10B_expr       224
ZFP42_expr        224
ZIC3_expr         224
ZSCAN18_expr      224
ADAM33_methy      224
AK123759_methy    224
ALDH3A1 _methy    224
CDH22_methy       224
CHL1_methy        224
COL20A1_methy     224
COL23A1_methy     224
CYBRD1_methy      224
DPPA3_methy       224
EPHB3_methy       224
FOXD2_methy       224
HLX-AS1_me

In [64]:
len(df_omics.columns), len(df_omics_clean.columns)

(118, 61)

In [65]:
len(df_omics_clean)

224

In [66]:
df_omics_clean.isna().any()

AFP_expr          False
BMP7_expr         False
CDH1_expr         False
CDX2_expr         False
CER1_expr         False
DAZL_expr         False
DNMT3B_expr       False
DNMT3L_expr       False
FGF4_expr         False
FOSL1_expr        False
HAND1_expr        False
IGF2_expr         False
JARID2_expr       False
KCNQ2_expr        False
LEFTY_expr        False
LUM_expr          False
MMP9_expr         False
MYC_expr          False
NANOG_expr        False
NESTIN_expr       False
NKX2-5_expr       False
OCT4_expr         False
OTX2_expr         False
PRDM14_expr       False
SALL4_expr        False
SOX2_expr         False
TBX3_expr         False
TFAP2A_expr       False
TFCP2L1_expr      False
TGFBR2_expr       False
WNT10B_expr       False
ZFP42_expr        False
ZIC3_expr         False
ZSCAN18_expr      False
ADAM33_methy      False
AK123759_methy    False
ALDH3A1 _methy    False
CDH22_methy       False
CHL1_methy        False
COL20A1_methy     False
COL23A1_methy     False
CYBRD1_methy    

Show the number of features of each omics after cleaning nans.

In [67]:
df_omics_clean.columns.str.endswith('_expr').astype(int).sum(), df_omics_clean.columns.str.endswith('_methy').astype(int).sum()

(34, 27)

In [68]:
df_omics_clean.head()

,AFP_expr,BMP7_expr,CDH1_expr,CDX2_expr,CER1_expr,DAZL_expr,DNMT3B_expr,DNMT3L_expr,FGF4_expr,FOSL1_expr,...,NXRA8_methy,PIWIL1_methy,PLEKHH3_methy,STON2P1_methy,SULT1A1_methy,TFAP2A_methy,TMEM173_methy,TNNI3_methy,UNC45A_methy,ZNF662_methy
Cell ID,,,,,,,,,,,,,,,,,,,,,
BJ_2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,1,0,1,1,0,1,0,1,0,0
BJ_44,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,15.750956,...,0,0,1,0,0,1,0,0,0,1
BJ_11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,18.601503,...,0,0,1,0,0,1,0,0,0,0
BJ_14,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.713610,...,0,0,1,1,0,1,0,0,0,0
BJ_19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,15.831870,...,0,0,1,0,0,1,0,0,0,0


In [69]:
num_cells = len(df_omics_clean)
num_genes = len(df_omics_clean.columns)
num_omics = 2
num_nodes = num_cells + num_genes + num_omics

num_cells, num_genes, num_nodes

(224, 61, 287)

In [70]:
df_omics_clean.isna().any().any()

False

Create the hypergraph. First construct node names and offsets.

In [74]:
omics_names = ['Omics_Expr', 'Omics_Methy']
cell_names = df_omics_clean.index.values.tolist()
gene_names = list(df_omics_clean.columns)

In [75]:
node_names = omics_names + gene_names + cell_names
node_names

['Omics_Expr',
 'Omics_Methy',
 'AFP_expr',
 'BMP7_expr',
 'CDH1_expr',
 'CDX2_expr',
 'CER1_expr',
 'DAZL_expr',
 'DNMT3B_expr',
 'DNMT3L_expr',
 'FGF4_expr',
 'FOSL1_expr',
 'HAND1_expr',
 'IGF2_expr',
 'JARID2_expr',
 'KCNQ2_expr',
 'LEFTY_expr',
 'LUM_expr',
 'MMP9_expr',
 'MYC_expr',
 'NANOG_expr',
 'NESTIN_expr',
 'NKX2-5_expr',
 'OCT4_expr',
 'OTX2_expr',
 'PRDM14_expr',
 'SALL4_expr',
 'SOX2_expr',
 'TBX3_expr',
 'TFAP2A_expr',
 'TFCP2L1_expr',
 'TGFBR2_expr',
 'WNT10B_expr',
 'ZFP42_expr',
 'ZIC3_expr',
 'ZSCAN18_expr',
 'ADAM33_methy',
 'AK123759_methy',
 'ALDH3A1 _methy',
 'CDH22_methy',
 'CHL1_methy',
 'COL20A1_methy',
 'COL23A1_methy',
 'CYBRD1_methy',
 'DPPA3_methy',
 'EPHB3_methy',
 'FOXD2_methy',
 'HLX-AS1_methy',
 'KCNQ2_methy',
 'LTBR_methy',
 'LY86-AS1_methy',
 'NFATC1_methy',
 'NFIX_methy',
 'NXRA8_methy',
 'PIWIL1_methy',
 'PLEKHH3_methy',
 'STON2P1_methy',
 'SULT1A1_methy',
 'TFAP2A_methy',
 'TMEM173_methy',
 'TNNI3_methy',
 'UNC45A_methy',
 'ZNF662_methy',
 'BJ_2

Save the node names.

In [76]:
node_file = savedir / 'nodes.csv'
node_file.write_text("\n".join(node_names))

2347

In [77]:
def get_omics_id_from_gene(gene_name: str):
    if gene_name.endswith('_expr'):
        return 0
    if gene_name.endswith('_methy'):
        return 1
    raise ValueError(gene_name)


Construct Hyperedges.

In [78]:
# Node offsets
omics_offset = 0
gene_offset = num_omics
cell_offset = num_omics + num_genes

In [79]:
hyper_edges = []
df = df_omics_clean

for cell_id in range(num_cells):
    for gene_id in range(num_genes):
        weight = df.iloc[cell_id, gene_id]
        if np.isnan(weight):
            continue

        omics_id = get_omics_id_from_gene(gene_names[gene_id])
        omics_node = omics_id + omics_offset
        gene_node = gene_id + gene_offset
        cell_node = cell_id + cell_offset
        edge = dict(Omics=omics_node, Gene=gene_node,
                    Cell=cell_node, Weight=weight)
        hyper_edges.append(edge)

df_edges =  pd.DataFrame.from_records(hyper_edges)

Save the hyperedges.

In [80]:
edge_file = savedir / 'edges.csv'
df_edges.to_csv(edge_file, index=False)

In [81]:
cell_stage = np.array(pd.read_csv("data/sc_GEM/cell_stage.csv", header=None))[0]
labels = []
for each in cell_stage:
    if each == "BJ":
        labels.append(0)
    if each == "d8":
        labels.append(1)
    if each == "d16T-" or each == "d16T+":
        labels.append(2)
    if each == "d24T-" or each == "d24T+":
        labels.append(3)
    if each == "IPS":
        labels.append(4)
    if each == "ES":
        labels.append(5)
cell_type = labels

Save the cell labels.

In [82]:
label_file = savedir / 'labels.csv'
label_file.write_text("\n".join(map(str, labels)))

447

Save the metadata.

In [83]:
metadata = dict(
    num_cells=num_cells, num_genes=num_genes, num_omics=num_omics,
    num_nodes=num_nodes, num_edges=len(hyper_edges),
    omics_offset=omics_offset, gene_offset=gene_offset, cell_offset=cell_offset,
)
metadata

{'num_cells': 224,
 'num_genes': 61,
 'num_omics': 2,
 'num_nodes': 287,
 'num_edges': 13664,
 'omics_offset': 0,
 'gene_offset': 2,
 'cell_offset': 63}

In [84]:
import json

metadata_file = savedir / 'metadata.json'
json.dump(metadata, metadata_file.open('w'))